In [1]:
import sys
from pathlib import Path

# 🔑 Automatically locate project root by searching upward for the 'src' folder
project_root = Path.cwd()
while not (project_root / "src").exists() and project_root != project_root.parent:
    project_root = project_root.parent

# Insert at the front of sys.path so Python finds 'src' first
sys.path.insert(0, str(project_root))
print(f"✅ Project root resolved: {project_root}")

✅ Project root resolved: c:\Users\igorp\OneDrive\Документы\GitHub\futures_trading_strategies


In [2]:
from src.data import load_simple_price_csv
from src.core import Asset, Capital, FixedRiskSizer
from src.engine import BacktestRunner
from src.strategies import BuyAndHoldStrategy

In [3]:
COMISSION = 0.0004
SLIPPAGE = 0.001
data1 = load_simple_price_csv("../data/MCFTR.csv")
data2 = load_simple_price_csv("../data/GLDRUB_TOM.csv")
MCFTR = Asset(ticker= 'MCFTR', price_data=data1, commission_rate=COMISSION, slippage_rate=SLIPPAGE)
GOLD = Asset(ticker= 'GLDRUB', price_data=data2, commission_rate=COMISSION, slippage_rate=SLIPPAGE)

In [4]:
INITIAL_CAPITAL = 100_000
VOLATILITY_WINDOW = 252
RISK_TARGET = 0.1
MAX_LEVERAGE = 1.0

In [5]:
risk_sizer = FixedRiskSizer(
    volatility_window=VOLATILITY_WINDOW, 
    risk_target=RISK_TARGET, 
    expected_sharpe=1, 
    max_leverage=MAX_LEVERAGE
)

# 2. Define Capital
capital = Capital(initial_capital=INITIAL_CAPITAL)

# 3. Initialize Runner and Run
runner1 = BacktestRunner(capital=capital, asset=MCFTR, sizer=risk_sizer)
runner2 = BacktestRunner(capital=capital, asset=GOLD, sizer= risk_sizer)
stock_result = runner1.run(BuyAndHoldStrategy())
gold_result = runner2.run(BuyAndHoldStrategy())

In [8]:
stock_result.print_summary()


════════════════════════════════════════════════════════════
  STRATEGY SUMMARY: Buy and Hold
════════════════════════════════════════════════════════════
  Total Return:      +142.40%
  CAGR:              +6.25%
  Gross Return:      +146.73%
------------------------------------------------------------
  Annual Volatility: 11.05%
  Max Drawdown:      -32.23%
  Avg Drawdown:      -2.12%
------------------------------------------------------------
  Sharpe Ratio:      0.57
  Gross Sharpe:      0.58
  Sortino Ratio:     0.66
------------------------------------------------------------
  Total Fees:        $4,333.42
  Fee Drag Ratio:    0.03
════════════════════════════════════════════════════════════



In [7]:
from src.visualization import print_comparison_table
